### requires_grad 의미 파악

In [4]:
import torch

# leaf 텐서 생성
x = torch.ones(3, requires_grad=False)

# requires_grad_() 메서드를 사용하여 자동 미분 추적 시작
x.requires_grad_(True)

print(x.requires_grad)  # 출력: True

True


In [3]:
import torch
x = torch.ones(1, requires_grad=True)  # 자동 미분 추적 
y = x * 3
print(x.grad)  # tensor([3.])
y.backward() # 역전파 수행 
print(x.grad)  # tensor([3.])
print(x.grad_fn)  # tensor([3.])

None
tensor([3.])
None


In [15]:
import torch

# leaf tensor
x = torch.tensor([1.,2.,3.], requires_grad=True)
y = x * 2        # non-leaf tensor
z = y.sum()

z.backward()      # gradient 계산

print(x.grad)     # tensor([2., 2., 2.])
print(z.grad_fn)

tensor([2., 2., 2.])


In [23]:
m1 = torch.FloatTensor([[1, 2], [3, 4]]).requires_grad_()
m2 = torch.FloatTensor([[1], [2]]).requires_grad_()

y = (m1.mul(m2))   # element-wise 곱
z = y.sum()        # 스칼라
print(y.grad_fn)   # AddBackward0 같은 함수가 출력됨 (연산 이력 확인)

z.backward()       # 기울기 계산
print(m1.grad)     # d(z)/d(m1)
print(m2.grad)     # d(z)/d(m2)


tensor([[1., 1.],
        [2., 2.]])
tensor([[3.],
        [7.]])


In [ ]:
import torch

# leaf 텐서 생성
x = torch.ones(3, requires_grad=True)

# 연산을 통해 non-leaf 텐서 생성
y = x + 2

# non-leaf 텐서에 대해 requires_grad_() 메서드 사용 시 오류 발생
try:
    y.requires_grad_(True)
except RuntimeError as e:
    print(f"오류 발생: {e}")

오류 발생: you can only change requires_grad flags of leaf variables. If you want to use a computed variable in a subgraph that doesn't require differentiation use var_no_grad = var.detach().


In [2]:
tensor1 = torch.tensor(1., requires_grad=True)
tensor2 = torch.tensor(2., requires_grad=True)
tensor3 = torch.tensor(3., requires_grad=True)
output_tensor = torch.mul(tensor1, tensor2)
output_tensor = torch.mul(output_tensor, tensor3)
output_tensor.backward()

print(f'tensor1.grad: {tensor1.grad}')
print(f'tensor2.grad: {tensor2.grad}')
print(f'tensor3.grad: {tensor3.grad}')
print(f'output_tensor.grad: {output_tensor.grad}')

tensor1.grad: 6.0
tensor2.grad: 3.0
tensor3.grad: 2.0
output_tensor.grad: None


/tmp/ipykernel_882/963793391.py:11: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more informations. (Triggered internally at /opt/conda/conda-bld/pytorch_1729647176074/work/build/aten/src/ATen/core/TensorBody.h:489.)
  print(f'output_tensor.grad: {output_tensor.grad}')


In [ ]:
tensor1 = torch.tensor(1., requires_grad=False)
tensor2 = torch.tensor(2., requires_grad=True)
tensor3 = torch.tensor(3., requires_grad=False)

output_tensor = torch.mul(tensor1, tensor2)
output_tensor = torch.mul(output_tensor, tensor3)

output_tensor.backward()

print(f'tensor1.grad: {tensor1.grad}')
print(f'tensor2.grad: {tensor2.grad}')
print(f'tensor3.grad: {tensor3.grad}')
print(f'output_tensor.grad: {output_tensor.grad}')

tensor1.grad: None
tensor2.grad: 3.0
tensor3.grad: None
output_tensor.grad: None


/tmp/ipykernel_882/239290478.py:11: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more informations. (Triggered internally at /opt/conda/conda-bld/pytorch_1729647176074/work/build/aten/src/ATen/core/TensorBody.h:489.)
  print(f'output_tensor.grad: {output_tensor.grad}')


backward() Function를 통해서 역전파를 시켜주게 되면 Tensor 의 grad 에 값이 할당

requires_grad=false 인 경우 grad 값이 None으로 고정 True 인 경우 자동 미분

- [참고1](https://westlife0615.tistory.com/866)
- [참고2](https://nuguziii.github.io/dev/dev-003/)


### Autograd 체험

In [4]:
x1 = torch.tensor(2.0, requires_grad=True)
x2 = torch.tensor(3.0, requires_grad=True)
z = x1**2 + x2**3 + x1*x2

print(f"함수 z = x1^2 + x2^3 + x1*x2")
print(f"x1={x1.item()}, x2={x2.item()}, z={z.item()}")

z.backward()
print(f"Autograd = {x1.grad} (이론값: 2*x1 + x2 = {2*2 + 3})")
print(f"Autograd = {x2.grad} (이론값: 3*x2^2 + x1 = {3*3**2 + 2})")

함수 z = x1^2 + x2^3 + x1*x2
x1=2.0, x2=3.0, z=37.0
Autograd = 7.0 (이론값: 2*x1 + x2 = 7)
Autograd = 29.0 (이론값: 3*x2^2 + x1 = 29)


### 모델 설계

In [5]:
import torch.nn as nn

In [6]:
class SimpleLinearModel(nn.Module):
    def __init__(self, input_size, output_size):
        super(SimpleLinearModel, self).__init__()
        self.linear = nn.Linear(input_size, output_size)
    
    def forward(self, x):
        return self.linear(x)

In [7]:
model = SimpleLinearModel(input_size=10, output_size=1)

In [8]:
x = torch.randn(32, 10)
output = model(x)
print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")

Input shape: torch.Size([32, 10])
Output shape: torch.Size([32, 1])


In [9]:
print(f"\nModel parameters:")
for name, param in model.named_parameters():
    print(f"{name}: {param.shape}")


Model parameters:
linear.weight: torch.Size([1, 10])
linear.bias: torch.Size([1])


### GPU 활용

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [12]:
tensor_cpu = torch.randn(3, 3)
tensor_gpu = tensor_cpu.to(device)
print('tensor_cpu: ',tensor_cpu.device)
print('tensor_gpu: ',tensor_gpu.device)

model = model.to(device)
print('model dvice: ',next(model.parameters()).device)

tensor_cpu:  cpu
tensor_gpu:  cpu
model dvice:  cpu


### 기본 훈련 루프

커스텀 데이터 셋

In [13]:
import pandas as pd
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

class CustomDataset(Dataset):
    def __init__(self, csv_file, device):
        self.data = pd.read_csv(csv_file)
        self.features = self.data.iloc[:, :-1].values
        self.labels = self.data.iloc[:, -1].values

        if self.labels.min() != 0:
            self.labels = self.labels - self.labels.min()

        self.device = device

    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, index):
        x = torch.tensor(self.features[index], dtype=torch.float32).to(self.device)
        y = torch.tensor(self.labels[index], dtype=torch.long).to(self.device)
        return x, y

In [14]:
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(MLP, self).__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        return self.layers(x)

In [15]:
dataset = CustomDataset('../../data/covtype.csv', device=device)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

In [1]:
input_dim = dataset.features.shape[1]
output_dim = len(set(dataset.labels))
hidden_dim = 64

NameError: name 'dataset' is not defined

In [ ]:
model = MLP(input_dim, hidden_dim, output_dim).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer=optimizer,lr_lambda=lambda epoch:0.95 ** epoch)

epochs = 5
for epoch in range(epochs):
    for x, y in dataloader:
        # (1) 예측
        y_pred = model(x)

        # (2) 손실 계산
        loss = criterion(y_pred, y)

        # (3) 그래디언트 초기화
        optimizer.zero_grad()

        # (4) 역전파
        loss.backward()

        # (5) 가중치 업데이트
        optimizer.step()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")